# Лабораторная работа 1 — рабочий ноутбук

**ФИО: Диденко Никита Валерьевич**

**Файлы данных:**

Этот ноутбук — чистовой рабочий документ, а не черновик. Он должен
исполняться сверху вниз без ошибок на выданном вам **сыром** файле
и на выходе давать очищенную таблицу и `cleaning_log.csv`.

Разделы ниже обязательны, порядок менять не нужно. Внутри раздела
добавляйте столько ячеек, сколько нужно. Если какой-то раздел у вас
пустой — напишите в нём одной строкой, почему.

## 0. Окружение и настройки

In [47]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

RAW_CLIENTS = "clients.csv"
RAW_TXN = "transactions.csv"

## 1. Загрузка сырых данных

Файлы выгружены разными системами. Убедитесь, что они прочитались
**правильно**: кодировка, разделитель, десятичный разделитель, типы,
даты, идентификаторы.

In [55]:
cli = pd.read_csv(RAW_CLIENTS)
cli

,client_id,birth_date,gender,city,segment,loyalty_tier,card_type,registration_date,card_open_date,income_source,monthly_income,has_mobile_app,app_last_login,satisfaction_score,branch_code,marketing_opt_in
0,C001526,1963-05-16,М,Санкт-Петербург,Массовый,Базовый,Кредитная,15.03.2015,06.03.2018,Зарплата в нашем банке,58900.0,Нет,NaN,3.0,335,Да
1,C002932,1955-05-17,М,Нижний Новгород,Массовый,Базовый,Дебетовая,23.05.2011,09.03.2016,Зарплата в нашем банке,53700.0,Да,2024-01,4.0,713,Да
2,C001462,1994-10-08,М,Санкт-Петербург,Зарплатный,Серебро,Дебетовая,25.04.2020,22.04.2025,Зарплата в нашем банке,54500.0,Нет,NaN,1.0,279,Нет
3,C001683,1964-09-22,Ж,Новосибирск,Зарплатный,Базовый,Дебетовая,03.07.2014,16.01.2017,Не указан,84400.0,Да,2024-02,4.0,619,Да
4,C005419,1978-08-16,М,Нижний Новгород,Массовый,Базовый,Дебетовая,08.08.2014,06.02.2019,Зарплата в другом банке,47300.0,Да,2024-03,3.0,636,Да
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5728,C002367,1962-01-01,М,Казань,Массовый,Серебро,Дебетовая,06.03.2013,05.01.2016,Не указан,69600.0,Нет,NaN,NaN,279,Да
5729,C005262,1985-10-07,Ж,Екатеринбург,Private,Платина,Премиальная,01.04.2012,06.02.2018,Зарплата в другом банке,254500.0,Нет,NaN,5.0,686,Да
5730,C005700,1978-12-20,Ж,Екатеринбург,Массовый,Базовый,Дебетовая,16.10.2020,27.03.2022,Зарплата в нашем банке,60500.0,Нет,NaN,NaN,926,Да
5731,C004245,1969-12-07,Ж,Москва,Премиальный,Серебро,Премиальная,08.12.2020,20.10.2021,Зарплата в нашем банке,122500.0,Да,2024-07,NaN,150,Да


In [57]:
txn = (pd.read_csv(RAW_TXN, encoding="cp1251", sep=";"))
txn

,txn_id,client_id,txn_datetime,amount,currency,category,mcc_code,channel,status
0,990050,C003010,2023-12-15,"473,34",RUB,Связь и интернет,4814,Карта,успешно
1,990083,C003001,2023-12-17,"167,44",RUB,Транспорт,4111,СБП,успешно
2,990039,C003369,2023-12-18 19:14:40,"543,82",RUB,Онлайн-сервисы,5815,Карта,успешно
3,990066,C000102,2023-12-23 13:30:46,"208,58",RUB,Аптеки,5912,Карта,успешно
4,990187,C000674,2023-12-23 14:13:36,"292,33",RUB,Связь и интернет,4814,Карта,успешно
...,...,...,...,...,...,...,...,...,...
625599,990019,C000454,2026-06-28 18:56:17,"15060,84",RUB,Одежда и обувь,5651,Токен,успешно
625600,990108,C005043,2026-07-01 13:35:35,"7949,41",RUB,Одежда и обувь,5651,Карта,успешно
625601,990191,C005162,2026-07-25 10:39:10,"1867,39",RUB,Развлечения и досуг,7832,Онлайн-банк,успешно
625602,990199,C004239,2026-08-02,"70,13",RUB,Транспорт,4111,Карта,успешно


## 2. Аудит качества

Здесь вы ничего не исправляете — только находите и измеряете.
Для каждой метрики качества заранее определите знаменатель.

### 2.1. Объём, типы, ключи, единица наблюдения

In [60]:
overview = pd.DataFrame({
    #'n_unique': cli.shape(),
    'dtype': cli.dtypes.astype(str),
    'n_unique': cli.nunique(),
    'n_missing': cli.isna().sum(),
    'missing_%': (cli.isna().mean() * 100).round(2),

})
overview

,dtype,n_unique,n_missing,missing_%
client_id,str,5678,0,0.00
birth_date,str,4838,0,0.00
gender,str,2,0,0.00
city,str,17,0,0.00
segment,str,4,0,0.00
loyalty_tier,str,4,0,0.00
card_type,str,3,0,0.00
registration_date,str,3281,0,0.00
card_open_date,str,2655,0,0.00
income_source,str,3,0,0.00


In [59]:
cli.info()

<class 'pandas.DataFrame'>
RangeIndex: 5733 entries, 0 to 5732
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   client_id           5733 non-null   str    
 1   birth_date          5733 non-null   str    
 2   gender              5733 non-null   str    
 3   city                5733 non-null   str    
 4   segment             5733 non-null   str    
 5   loyalty_tier        5733 non-null   str    
 6   card_type           5733 non-null   str    
 7   registration_date   5733 non-null   str    
 8   card_open_date      5733 non-null   str    
 9   income_source       5733 non-null   str    
 10  monthly_income      5199 non-null   float64
 11  has_mobile_app      5733 non-null   str    
 12  app_last_login      3525 non-null   str    
 13  satisfaction_score  2267 non-null   float64
 14  branch_code         5733 non-null   int64  
 15  marketing_opt_in    5733 non-null   str    
dtypes: float64(2), in

### 2.2. Дубли

### 2.3. Соответствие данных описанию полей из задания

### 2.4. Связь между таблицами

**Выводы по аудиту.** Перечислите найденное: что это, сколько строк,
насколько это мешает решить задачу.

## 3. Препроцессинг

На каждое изменение — строка в журнале (раздел 4). Решение «оставить
как есть» тоже является решением и тоже попадает в журнал.

### 3.1. Типы, даты, числа

### 3.2. Категориальные признаки

### 3.3. Пропуски

Для каждого поля с пропусками: доля, предположение о механизме, решение.

### 3.4. Выбросы

Обнаружение и решение — это два разных шага. Сначала первый, потом второй.

### 3.5. Фильтры, применяемые перед расчётом активности

Покажите воронку: сколько строк на входе и сколько осталось после каждого фильтра.

## 4. Журнал изменений

Одна строка — одно решение, а не одна запись. Колонки: `field`, `issue`,
`scope`, `n_rows`, `value_before`, `action`, `value_after`, `rationale`.

In [50]:
cleaning_log = pd.DataFrame(columns=[
    "field", "issue", "scope", "n_rows",
    "value_before", "action", "value_after", "rationale",
])

## 5. Разведочный анализ

Графиков столько, сколько нужно. В отчёт пойдут не все.

### 5.1. Распределения количественных признаков

### 5.2. Частотные характеристики категориальных признаков

### 5.3. Динамика по месяцам

### 5.4. Разрезы и связи между признаками

## 6. Метрика активности

Что именно вы считаете активностью, как она вычисляется, на каком
множестве клиентов, за какие окна. Отдельно: какое снижение эта метрика
**не увидит**.

## 7. Поиск и описание групп

Правило отбора для каждой группы, её размер, причина падения.
Правило разрешения конфликтов, если клиент подходит под несколько групп.

### 7.1. Кого в кампанию не берём и почему

## 8. Что проверили и что не подтвердилось

Гипотезы, которые вы проверили и отвергли. По одной строке на гипотезу,
с числом.

## 9. Сохранение результатов

Три файла: очищенная таблица для второго ноутбука, журнал изменений,
итоговый список кампании.

In [51]:
# clean.to_parquet("clean_data.parquet", index=False)
# cleaning_log.to_csv("cleaning_log.csv", index=False, encoding="utf-8")
# campaign[["client_id", "group"]].to_csv("campaign_list.csv", index=False, encoding="utf-8")